### 1. Jev 主要支持以下三种模式（Primitive 问答类型）

官方文档参考：[TypeSafe](https://docs.typesafe.ai/confidence)

| 原语类型 (Question type) | 核心目标 (Goal) | 返回字段 (Returns) | 取值范围与核心特点                                                                                       |
| :--- | :--- | :--- |:------------------------------------------------------------------------------------------------|
| **`Choice`**（单选分类模式） | 从一组**无序**候选字典中选择最匹配的一项 | `choice`, `probabilities`, `confidence` | `confidence` 归一化在 `0.0 ~ 1.0`                                                                   |
| **`Noul`**（是非二分模式） | 对一个陈述判断是否为真（Yes / No） | `noul` | 返回 `0.0 ~ 1.0` 之间“为真”的概率（无单独 `confidence`，越靠近 0 或 1 越确定，`0.5` 表示完全拿不准）                          |
| **`Score`**（量表评分模式） | 根据从低到高的**有序**等级列表（共 $n$ 档）进行打分 | `score`, `probabilities`, `confidence` | `score` **未做 0~1 归一化**，范围是 **`0.0 ~ n-1`**（等级期望值）；`confidence` 归一化在 `0.0 ~ 1.0`，会额外惩罚跨等级的“两极撕裂” |

### 2. Jev (TypeSafe SDK) 三种核心模式 API 用法示例

In [4]:
from dotenv import load_dotenv

load_dotenv()
from typesafe_sdk import TypeSafeClient, Choice, Noul, Score

with TypeSafeClient() as client:
    # 模拟一段用户客服反馈文本作为输入状态
    customer_message = (
        "我这个月的 Pro 会员被重复扣款了两次！"
        "请立刻帮我把多扣的钱退掉，这已经是本周第二次出现系统故障了，非常愤怒！"
    )

    response = client.system_one(
        state={"message": customer_message},
        questions={
            # 1. Choice 模式：分类选择题（多选一，附带各选项概率及置信度）
            "department": Choice(
                instructions="应该将此消息分派给哪个团队处理？",
                criteria={
                    "billing": "涉及扣款、退款、账单或支付异常的问题",
                    "technical": "涉及应用崩溃、功能失效或系统故障的Bug",
                    "general": "其他普通咨询或账号建议"
                }
            ),

            # 2. Noul 模式：是非判断题（返回 0~1 的布尔成立概率）
            "is_angry": Noul(
                instructions="客户在文本中表现出了明显的愤怒或情绪失控？"
            ),

            # 3. Score 模式：量表评分题（根据指定的等级从低到高计算加权得分）
            "urgency": Score(
                instructions="评估该工单的紧急程度",
                criteria=[
                    "不紧急：常规询问，不影响日常使用",
                    "一般：功能受损但有临时替代方案",
                    "紧急：核心资产受损（如重复扣款/服务完全中断），需立刻人工介入"
                ]
            ),
        },
    )

### 3. 读取和解析返回结果

#### 3.1 读取 Choice 结果与置信度计算

- **Choice 置信度公式**：
  $$\text{confidence}_{\text{choice}} = \frac{p_{\max} - \frac{1}{n}}{1 - \frac{1}{n}}$$
  假设 $n=3$ 个部门，当 `billing` 概率为 `1.0` 时，$\text{confidence} = \frac{1.0 - 1/3}{1 - 1/3} = 1.0$；若三个部门概率平摊为 `(0.33, 0.33, 0.33)`，则 `confidence = 0.0`。

In [2]:
dept_answer = response.choices["department"]
print(f"【Choice 分类模式】")
print(f"  - 最佳选择: {dept_answer.choice}")
print(f"  - 置信度 (Confidence): {dept_answer.confidence}")
print(f"  - 完整概率分布 (Probabilities): {dept_answer.probabilities}")

【Choice 分类模式】
  - 最佳选择: billing
  - 置信度 (Confidence): 1.0
  - 完整概率分布 (Probabilities): {'billing': 1.0, 'technical': 0.0, 'general': 0.0}


#### 3.2 读取 Noul 结果

- **为什么没有单独的 `confidence` 字段？** 因为在是非题中，概率 $p$（`noul`）本身就代表了确定性：越接近 `1.0`（确信为真）或 `0.0`（确信为假）越确定，接近 `0.5` 说明模型拿不准（若需换算成 `0~1` 置信度，公式即为 $|2p - 1|$）。

In [5]:
angry_answer = response.nouls["is_angry"]
print(f"【Noul 是非模式】")
print(f"  - 为真的概率 (Probability): {angry_answer.noul}")
# 通常通过设定阈值转为布尔动作
is_customer_angry = angry_answer.noul > 0.8
print(f"  - 阈值判定 (>0.8): {'触发情绪安抚流程' if is_customer_angry else '正常处理'}")

【Noul 是非模式】
  - 为真的概率 (Probability): 0.01
  - 阈值判定 (>0.8): 正常处理


#### 3.3 读取 Score 结果（深度解析：`score` 与 `confidence` 的计算与分工）

在使用 `Score` 原语时，有三个最核心的问题需要理清：

##### 问题 1：`score` 得分是怎么算出来的？为什么范围不是 `0 ~ 1`？
- `score` **没有做 `0 ~ 1` 归一化**！假设 `criteria` 定义了 $n$ 个有序等级，系统会自动将其编号为 `0` 到 `n - 1`（本例中 `urgency` 有 3 档：`0=不紧急, 1=一般, 2=紧急`，所以得分范围是 **`0.0 ~ 2.0`**）。
- `score` 的本质就是**等级编号 $i$ 的概率加权平均值（数学期望）**：
  $$\text{score} = \sum_{i=0}^{n-1} i \times p_i$$
  本例中概率分布为 `{0: 0.0, 1: 0.0, 2: 1.0}`，因此 $\text{score} = 0\times 0 + 1\times 0 + 2\times 1.0 = \mathbf{2.0}$。

##### 问题 2：为什么 `Score` 的 `confidence`（置信度）计算公式比 `Choice` 复杂？
- 因为 `Score` 的等级是**有序**的！**“模型在相邻的 1 分和 2 分之间纠结”**，要比 **“模型在极端的 0 分（不紧急）和 2 分（紧急）之间两极撕裂”** 确定得多。
  - **相邻纠结 `{0: 0.0, 1: 0.5, 2: 0.5}`**：此时 $\text{confidence} = 1 - \frac{0.5}{2/3} = \mathbf{0.25}$。
  - **两极撕裂 `{0: 0.5, 1: 0.0, 2: 0.5}`**：此时 $\text{confidence} = \max(0, 1 - \frac{1.0}{2/3}) = \mathbf{0.0}$！
用“打靶”来理解这两种差异：

相邻纠结（例如 1 分和 2 分各占 50%）
场景：“我的页面一直报错，赶紧处理！”（模型确定不是 0 分，但拿不准算 1 分还是 2 分）。

例子：子弹打在了 8 环和 9 环。大方向没歪，只是在邻近刻度摇摆。

结果：模型认为虽然拿不准，但还可以给一点点置信度（例如 0.25）

两极撕裂（例如 0 分和 2 分各占 50%）

场景：“呵呵，你们系统可真是太聪明了呢（反讽：模型觉得 50% 是真心夸奖即 0 分，50% 是极度愤怒投诉即 2 分）。

例子：一枪打在 0 环，一枪打在 10 环，两极开花，完全“精神分裂”。

结果：你完全不知道这到底是件好事（0分）还是惊天大雷（2分），模型会一眼识破这种混乱，认为这完全不可信，把置信度直接强行归零（0.0）。

一句话在有序的尺子上，晃动的跨度越小（相邻纠结），结果越靠谱；晃动的跨度越大（两极撕裂），结果越不可信。

##### 问题 3：写业务代码时，到底是用 `score` 还是用 `confidence`？
它们不是“二选一”，而是**“先后配合”**的关系：
- **`confidence` 是“守门员”**（回答：*模型看懂这道题了吗？给的分数靠谱吗？*）
- **`score` 是“业务结论”**（回答：*工单到底有多紧急？*）

**为什么不能只看 `score`？** 下面三种情况算出来的 `score` **全都是 `1.0`**：

| 场景 | 概率分布 `{0, 1, 2}` | 加权得分 `score` | 置信度 `confidence` | 真实含义与推荐系统动作 |
| :--- | :--- | :---: | :---: | :--- |
| **笃定为一般** | `{0: 0.0, 1: 1.0, 2: 0.0}` | **`1.0`** | **`1.00`** | 模型 100% 确定属于第 1 档（一般紧急），按常规流程处理。 |
| **完全抓瞎** | `{0: 0.33, 1: 0.33, 2: 0.33}` | **`1.0`** | **`0.00`** | 模型完全没看懂，三个等级平分概率，均值恰好为 `1.0`。 |
| **两极撕裂** | `{0: 0.50, 1: 0.0, 2: 0.50}` | **`1.0`** | **`0.00`** | 用户可能在反讽/阴阳怪气，模型认为 50% 是没事（`0`），50% 是严重危机（`2`），平均下来也是 `1.0`！ |

**实战用法总结**：
1. **实时分流/拦截**：先看 `confidence` 守门，通过后再看 `score` 分流。

In [6]:
urgency_answer = response.scores["urgency"]
print(f"【Score 评分模式】")
print(f"  - 计算出的加权得分: {urgency_answer.score}")
print(f"  - 各等级的概率细分: {urgency_answer.probabilities}")
print(f"  - 置信度: {urgency_answer.confidence}")

【Score 评分模式】
  - 计算出的加权得分: 2.0
  - 各等级的概率细分: {0: 0.0, 1: 0.0, 2: 1.0}
  - 置信度: 1.0
